<a href="https://colab.research.google.com/github/tejsvinimeshram/Practical/blob/main/EE524_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

DATA_PATH = "/content/motor_sensor_dataset.csv"

In [ ]:
# 1. Load data
df = pd.read_csv(DATA_PATH)

feature_cols = [
    "current_rms_A",
    "vibration_rms_mm_s",
    "temperature_C",
    "voltage_unbalance_pct",
    "speed_rpm",
    "bearing_freq_amplitude",
    "stator_freq_amplitude"
]

X = df[feature_cols].values
y_raw = df["fault_label"].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_

In [ ]:
# 2. Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=0.2, random_state=42, stratify=y )

In [ ]:
# 3. Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
results = {}

In [ ]:
# 4. Model A: Random Forest
rf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

# Tree models don't need scaling
y_pred_rf = rf.predict(X_test)
results["Random Forest"] = {
    "accuracy": accuracy_score(y_test, y_pred_rf),
    "precision": precision_score(y_test, y_pred_rf, average="macro"),
    "recall": recall_score(y_test, y_pred_rf, average="macro"),
    "f1": f1_score(y_test, y_pred_rf, average="macro"),
    "confusion_matrix": confusion_matrix(y_test, y_pred_rf),
    "report": classification_report(y_test, y_pred_rf, target_names=class_names)
}

In [ ]:
# 5. Model B: SVM (RBF kernel) on scaled features
svm = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
svm.fit(X_train_s, y_train)
y_pred_svm = svm.predict(X_test_s)
results["SVM (RBF)"] = {
    "accuracy": accuracy_score(y_test, y_pred_svm),
    "precision": precision_score(y_test, y_pred_svm, average="macro"),
    "recall": recall_score(y_test, y_pred_svm, average="macro"),
    "f1": f1_score(y_test, y_pred_svm, average="macro"),
    "confusion_matrix": confusion_matrix(y_test, y_pred_svm),
    "report": classification_report(y_test, y_pred_svm, target_names=class_names)
}

In [ ]:
# 6. Feature importance (from Random Forest)
importances = sorted(
    zip(feature_cols, rf.feature_importances_),
    key=lambda x: -x[1]
)

In [ ]:
# 7. Print summary
print("=" * 60)
for model_name, r in results.items():
    print(f"\nModel: {model_name}")
    print(f"  Accuracy : {r['accuracy']:.4f}")
    print(f"  Precision: {r['precision']:.4f}")
    print(f"  Recall   : {r['recall']:.4f}")
    print(f"  F1-score : {r['f1']:.4f}")
    print("  Confusion Matrix (rows=actual, cols=predicted):")
    print("  Classes:", list(class_names))
    print(r["confusion_matrix"])
    print("\n  Classification report:")
    print(r["report"])

print("\nFeature importance (Random Forest):")
for feat, imp in importances:
    print(f"  {feat:28s} {imp:.4f}")
print("=" * 60)


Model: Random Forest
  Accuracy : 0.9400
  Precision: 0.9399
  Recall   : 0.9400
  F1-score : 0.9399
  Confusion Matrix (rows=actual, cols=predicted):
  Classes: ['Bearing Fault', 'Healthy', 'Rotor Fault', 'Winding Fault']
[[92  3  5  0]
 [ 0 99  1  0]
 [ 8  1 89  2]
 [ 0  0  4 96]]

  Classification report:
               precision    recall  f1-score   support

Bearing Fault       0.92      0.92      0.92       100
      Healthy       0.96      0.99      0.98       100
  Rotor Fault       0.90      0.89      0.89       100
Winding Fault       0.98      0.96      0.97       100

     accuracy                           0.94       400
    macro avg       0.94      0.94      0.94       400
 weighted avg       0.94      0.94      0.94       400


Model: SVM (RBF)
  Accuracy : 0.9300
  Precision: 0.9299
  Recall   : 0.9300
  F1-score : 0.9297
  Confusion Matrix (rows=actual, cols=predicted):
  Classes: ['Bearing Fault', 'Healthy', 'Rotor Fault', 'Winding Fault']
[[90  4  6  0]
 [ 0 99  1 